# AWS Directory: /Users/jorge.rodriguez/Google Drive/My Drive/GitHub/FinOps_AWS
# GCP Directory: /Users/jorge.rodriguez/Google Drive/My Drive/GitHub/FinOps_GCP


## AWS vs GCP — Executive KPIs (trailing 24 months)
**Year 1** = months 24–13 back · **Year 2** = last 12 months. Both clouds are split into **Cloud Services** (first-party),
**Marketplace** (third-party software billed through the cloud) and **Tax**. GCP is net of credits.

Logic lives in `multicloud_kpis.py` (reuses `finops_exec_report.py` for AWS). Terminal: `python multicloud_kpis.py [--refresh-gcp]`

In [12]:
import importlib
import pandas as pd
import finops_exec_report as fr
import multicloud_kpis as mk
importlib.reload(fr); importlib.reload(mk)
pd.set_option('display.float_format', '{:,.0f}'.format)

print('AWS dir:', mk.AWS_DIR)
print('GCP dir:', mk.GCP_DIR)

AWS dir: /Users/jorge.rodriguez/Library/CloudStorage/GoogleDrive-jorge.rodriguez@audacy.com/My Drive/GitHub/FinOps_AWS
GCP dir: /Users/jorge.rodriguez/Library/CloudStorage/GoogleDrive-jorge.rodriguez@audacy.com/My Drive/GitHub/FinOps_GCP


## Parameters

In [13]:
END_MONTH = None          # e.g. '2026-09'; None = latest month both clouds have
AWS_BUDGET = 500_000      # AWS Cloud Services monthly budget (Marketplace excluded)
GCP_BUDGET = 300_000      # GCP Cloud Services monthly budget (Marketplace excluded)
REFRESH_GCP = False       # True = re-query BigQuery (needs `gcloud auth application-default login`, ~15 GB scanned)

## Load data

In [14]:
if REFRESH_GCP:
    print('Refreshed:', mk.refresh_gcp_history(END_MONTH))

df = mk.load_all()
kd = mk.compute(df, END_MONTH, AWS_BUDGET, GCP_BUDGET)
print(f"Year 1: {kd.y1[0]:%b %Y} – {kd.y1[1]:%b %Y}")
print(f"Year 2: {kd.y2[0]:%b %Y} – {kd.y2[1]:%b %Y}")
df.groupby(['Provider', 'Category'])['Cost'].sum().unstack()

Year 1: Oct 2024 – Sep 2025
Year 2: Oct 2025 – Sep 2026


Category,Cloud Services,Marketplace,Tax
Provider,,,
AWS,"32,368,623","12,013,579","1,135,185"
GCP,"4,512,739","1,603,834","193,806"


## Key findings

In [15]:
for line in kd.k['insights']:
    print('•', line)

• Combined cloud spend was $12.19M in the last 12 months vs $11.13M the year before (+9.5%). AWS -20.3%, GCP +178.9%.
• Cloud Services run-rate: AWS $439K → $459K/mo, GCP $92K → $347K/mo. GCP is now 75% the size of AWS (was 21%).
• GCP biggest increases (avg $/mo, Y2 vs Y1): VMware Engine +$40K (one-time charge), Gemini API +$33K, BigQuery +$24K.
• GCP biggest decreases: Cloud SQL -$2.2K, GKE Enterprise / GDC -$663.
• AWS biggest increases (avg $/mo, Y2 vs Y1): CloudFront +$13K, Direct Connect +$4.2K, CloudTrail +$3.8K.
• AWS biggest decreases: RDS -$28K, EC2 Compute -$21K.
• GenAI/ML spend: $73K → $525K (GCP $506K, AWS $18K in the last 12 months).
• Only 42% of taggable AWS spend and 11% of GCP spend carry an environment tag/label — cost allocation by environment is not possible today for most spend.
• Marketplace (third-party) was $3.26M in the last 12 months — $2.27M on AWS and $995K on GCP — outside both Cloud Services budgets.
• Largest one-time charges: GCP VMware Engine $478K (O

## KPI scorecard

In [16]:
from IPython.display import HTML
HTML(f"<style>{mk.CSS}</style>" + mk.scorecard_html(kd))

## Spend trend

In [17]:
mk.chart_monthly(kd).show()
mk.chart_runrate(kd).show()
mk.chart_quarterly(kd).show()

## Service movers (Year 2 vs Year 1, avg $/month)

In [18]:
mk.chart_movers(kd, mk.GCP).show()
mk.chart_movers(kd, mk.AWS).show()

In [19]:
display(kd.k['movers'][mk.GCP].sort_values('Δ/mo', ascending=False).head(15))
display(kd.k['movers'][mk.AWS].sort_values('Δ/mo', ascending=False).head(15))

,Y1 avg/mo,Y2 avg/mo,Δ/mo,one_time
Service,,,,
VMware Engine,0,"40,289","40,289",True
Gemini API,108,"32,689","32,581",False
BigQuery,"8,027","32,017","23,990",False
BigQuery Reservation API,"21,842","38,638","16,797",False
App Engine,"1,817","18,098","16,281",False
Support,"5,235","20,772","15,537",False
Cloud Storage,"5,202","19,913","14,712",False
Cloud Pub/Sub,"1,596","12,579","10,983",False
Compute Engine,"7,789","15,611","7,822",False


,Y1 avg/mo,Y2 avg/mo,Δ/mo,one_time
Service,,,,
Amazon CloudFront,"21,865","35,189","13,324",False
AWS Direct Connect,442,"4,596","4,154",False
AWS CloudTrail,"5,341","9,165","3,825",False
Amazon Elastic Container Service for Kubernetes,"2,390","3,524","1,134",False
AWS DataSync,32,654,622,False
AWSDevOpsAgent,0,281,281,True
Amazon Kendra,0,262,262,False
Amazon Route 53,"1,712","1,822",110,False
AWS Lambda,"2,618","2,711",93,False


## GenAI, tagging, one-time charges

In [20]:
mk.chart_genai(kd).show()
mk.chart_tagcov(kd).show()
kd.k['lumpy']

,Provider,Category,Service,Date,Cost
3059,GCP,Cloud Services,VMware Engine,2025-10-01,"477,852"
2708,GCP,Cloud Services,Gemini API,2026-01-01,"102,107"


# 2027 Forecast (built only from 2026 actuals)
* **Cloud Services:** start from the Jul–Sep 2026 exit run-rate; trend = Theil-Sen slope of 2026 months
  (AWS months with a missing Savings Plans fee are normalized for the trend).
  **Low** = flat · **Base** = damped trend (momentum fades 15%/month) · **High** = 2026 slope continues.
* **Marketplace:** each contract repeats its 2026 billing cadence (monthly / quarterly / annual) and last amount.
* Logic: `cloud_forecast.py`

In [ ]:
import cloud_forecast as cf
importlib.reload(cf)
fc = cf.forecast(kd.df, kd.budgets)        # kd.budgets = {'AWS': AWS_BUDGET, 'GCP': GCP_BUDGET}
print(f"Basis: {fc.basis_start:%b %Y} – {fc.basis_end:%b %Y}   Forecast: {fc.horizon[0]:%b %Y} – {fc.horizon[-1]:%b %Y}")

## Forecast executive summary

In [ ]:
import re
S = fc.summary
print(f"FY2027 Cloud Services (base): ${S['svc_total']['Base']:,.0f}  vs budget ${S['budget_total']:,.0f}")
print(f"FY2027 Marketplace:           ${S['mkt_total']:,.0f}")
print(f"FY2027 Combined (base):       ${S['combined']['Base']:,.0f}   (+ est. tax ${S['tax']:,.0f})")
for title, lines in S['paragraphs']:
    print(f'\n{title.upper()}')
    for line in lines:
        print('  •', re.sub('<[^>]+>', '', line))

In [ ]:
cf.chart_fy_bridge(fc).show()

## a) Cloud Services only (no Marketplace)

In [ ]:
cf.chart_services(fc, mk.AWS).show()
cf.chart_services(fc, mk.GCP).show()

In [ ]:
display(cf.services_budget_table(fc))          # quarterly Low / Base / High vs budget
display(cf.monthly_table(fc, 'services'))       # monthly base vs budget

In [ ]:
display(fc.drivers[mk.AWS].head(10))
display(fc.drivers[mk.GCP].head(10))

## b) Marketplace only (no Services)

In [ ]:
cf.chart_marketplace(fc).show()

In [ ]:
display(cf.quarterly_table(fc, 'marketplace'))
display(cf.monthly_table(fc, 'marketplace'))
display(fc.contracts)
display(fc.q4_risk)   # billed Q4 2025, not seen in 2026 -> NOT in the forecast

## c) Combined — Services + Marketplace

In [ ]:
cf.chart_combined(fc).show()

In [ ]:
display(cf.quarterly_table(fc, 'combined'))
display(cf.monthly_table(fc, 'combined'))

## Export the HTML report

In [21]:
print('Report written:', mk.build_report(kd, fc=fc))   # includes the 2027 forecast sections

Report written: /Users/jorge.rodriguez/Library/CloudStorage/GoogleDrive-jorge.rodriguez@audacy.com/My Drive/GitHub/FinOps_AWS/Cloud_KPI_Report_2026_09.html
